# Download sample origins — 2021, 2024 & 2026 snapshots

Reads `passing_origins_sample.parquet` and, for each repo, downloads **three independent
source-tree snapshots** using `swh-dataset-downloader`: the HEAD revision closest to end
of **2021**, closest to **2024**, and closest to **2026**, taken from the `yearly_head_revisions`
column.

Output layout:
```
output/sources/2021_<owner>_<repo>/
output/sources/2024_<owner>_<repo>/
output/sources/2026_<owner>_<repo>/
```

In [5]:
import logging
import os
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from dotenv import load_dotenv
from downloader.models import DownloadSource, TaskStatus
from downloader.pipeline import run_pipeline

# ── Paths ─────────────────────────────────────────────────────────────────────
PARQUET      = Path("../output-final/passing_origins_sample.parquet")  # adjust if needed
DOWNLOAD_DIR = Path("../output-final/sources")

# ── Tokens — loaded from python/.env (see .env.example) ──────────────────────
load_dotenv(Path(".") / ".env")
SWH_TOKEN    = os.environ.get("SWH_TOKEN", "")

#assert SWH_TOKEN, "Create python/.env from .env.example and set SWH_TOKEN"

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logging.getLogger("downloader").setLevel(logging.INFO)

In [6]:
df = pd.read_parquet(PARQUET)
print(f"Total passing origins: {len(df)}")
df[["url", "head_swhid", "top_language", "total_commits", "estimated_size_bytes", "yearly_head_revisions"]].head()

Total passing origins: 50


,url,head_swhid,top_language,total_commits,estimated_size_bytes,yearly_head_revisions
0,https://github.com/RockyLOMO/rxlib,swh:1:rev:5d9bc8d39c7cf18f4ffd3d31182c646ffeeb...,Java,3921,6454432,"[(2021-12-28, swh:1:rev:88b5264ab22dec7287d234..."
1,https://github.com/cgeo/cgeo,swh:1:rev:8f32a27bc6117ec4e5b0ba70d83bb19984ce...,Java,26137,76016432,"[(2021-12-30, swh:1:rev:509883af5b6db985cac340..."
2,https://github.com/fr3ts0n/AndrOBD,swh:1:rev:c945feca32c94b7148cdb791ff67631f52fb...,Java,1916,62117359,"[(2021-12-22, swh:1:rev:5549febfb5faddc7fa00e1..."
3,https://github.com/Zettelkasten-Team/Zettelkasten,swh:1:rev:77313b1fe087afcc449db6db85fa8ca925a4...,Java,1312,10325420,"[(2021-12-31, swh:1:rev:371175aaa4e38fc051655b..."
4,https://github.com/AdamaJava/adamajava,swh:1:rev:d15c117378c79724b28cbc661ccd428cc87c...,Java,6551,53006601,"[(2021-12-30, swh:1:rev:d001a2d1dcebcce979506d..."


In [7]:
def year_revision(yearly_head_revisions, year: str) -> str | None:
    """Return the SWHID of the head revision closest to the given year, if any."""
    for date, swhid in yearly_head_revisions:
        if date.startswith(year):
            return swhid
    return None


def repo_slug(url: str) -> str:
    """owner/repo → owner_repo, filesystem-safe."""
    return urlparse(url).path.strip("/").replace("/", "_")


checkpoints = pd.DataFrame(
    {
        "url": df["url"],
        "repo_slug": df["url"].map(repo_slug),
        "swhid_2021": df["yearly_head_revisions"].map(lambda r: year_revision(r, "2021")),
        "swhid_2024": df["yearly_head_revisions"].map(lambda r: year_revision(r, "2024")),
        "swhid_2026": df["yearly_head_revisions"].map(lambda r: year_revision(r, "2026")),
    }
)
missing = checkpoints[checkpoints["swhid_2021"].isna() | checkpoints["swhid_2024"].isna() | checkpoints["swhid_2026"].isna()]
print(f"{len(checkpoints)} repos, {len(missing)} missing a 2021, 2024 or 2026 checkpoint")
checkpoints

50 repos, 0 missing a 2021, 2024 or 2026 checkpoint


,url,repo_slug,swhid_2021,swhid_2024,swhid_2026
0,https://github.com/RockyLOMO/rxlib,RockyLOMO_rxlib,swh:1:rev:88b5264ab22dec7287d234726ecec26e9420...,swh:1:rev:ee74de713ebf246e533ff81321686a170824...,swh:1:rev:5d9bc8d39c7cf18f4ffd3d31182c646ffeeb...
1,https://github.com/cgeo/cgeo,cgeo_cgeo,swh:1:rev:509883af5b6db985cac340a5d9539ca6a541...,swh:1:rev:8f983a7b79e613d5e8f9be4f4d0402feecea...,swh:1:rev:8f32a27bc6117ec4e5b0ba70d83bb19984ce...
2,https://github.com/fr3ts0n/AndrOBD,fr3ts0n_AndrOBD,swh:1:rev:5549febfb5faddc7fa00e1d1d829a26d6a98...,swh:1:rev:7bf7133fb97298e654574db2d057e1a761fa...,swh:1:rev:c945feca32c94b7148cdb791ff67631f52fb...
3,https://github.com/Zettelkasten-Team/Zettelkasten,Zettelkasten-Team_Zettelkasten,swh:1:rev:371175aaa4e38fc051655b55703d66b99378...,swh:1:rev:b2b3484326be8fd2ab20c93dce255608ceff...,swh:1:rev:77313b1fe087afcc449db6db85fa8ca925a4...
4,https://github.com/AdamaJava/adamajava,AdamaJava_adamajava,swh:1:rev:d001a2d1dcebcce979506d5bec2efa91539b...,swh:1:rev:e3968a72f4761c2fa1be63ccee5ebd7c14fd...,swh:1:rev:d15c117378c79724b28cbc661ccd428cc87c...
5,https://github.com/ta4j/ta4j,ta4j_ta4j,swh:1:rev:8994ecbc9e644f7cfc36216c7de9eef4b65a...,swh:1:rev:ccaee65ff838acb79a397a76cfda98d078cd...,swh:1:rev:0f637ec08abc602916dd75a8cedc0a0491cd...
6,https://github.com/sakaiproject/sakai,sakaiproject_sakai,swh:1:rev:161305e1acded6039252a66bab2f088d4cc7...,swh:1:rev:9cda7021efa37ab95fead7552694970c2ec1...,swh:1:rev:a092dbf3dc6bf343131f50007c207a9abd95...
7,https://github.com/Iterable/iterable-android-sdk,Iterable_iterable-android-sdk,swh:1:rev:f0a694a31c29e37c0bc6156a1b414d73da9e...,swh:1:rev:8fdec83bb64a9ae02224fe4d19470631d324...,swh:1:rev:829bd6cff96b6279feae12df4472d9997138...
8,https://github.com/wso2/carbon-identity-framework,wso2_carbon-identity-framework,swh:1:rev:8ee11f7d29a0d84594927562299582fb7c1d...,swh:1:rev:f70492edb25c0139da35e7eb0db693033ed8...,swh:1:rev:463b223a8af6ad014ce2319132ce76c790ed...
9,https://github.com/GoogleCloudPlatform/Dataflo...,GoogleCloudPlatform_DataflowTemplates,swh:1:rev:9dd14da865f47f6ba33f0280c509b89a0482...,swh:1:rev:8e7ae4f91b0b30af3b055614d643595f1170...,swh:1:rev:0d4af552beccc59947190c61b3fb2d8f4073...


In [8]:
import downloader.pipeline as dl_pipeline

YEARS = ["2021", "2024", "2026"]

pairs = []
for row in checkpoints.dropna(subset=[f"swhid_{y}" for y in YEARS]).itertuples():
    for year in YEARS:
        pairs.append({
            "url": row.url,
            "swhid": getattr(row, f"swhid_{year}"),
            "output_dir": f"{year}_{row.repo_slug}",
        })

# run_pipeline ignores "output_dir" and names folders "<index>_<slug>_<hash>";
# override its naming so each folder is prefixed by the snapshot year instead.
dl_pipeline._make_output_dir_name = lambda index, url, swhid: pairs[index]["output_dir"]

print(f"{len(pairs)} downloads queued ({len(pairs) // 3} repos × 3 snapshots)")

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Jupyter supports top-level await — no asyncio.run() needed
results = await run_pipeline(
    pairs=pairs,
    output_base=DOWNLOAD_DIR,
    swh_token=SWH_TOKEN,
    debug=True,
)

13:12:59 [INFO] [0] start  https://github.com/RockyLOMO/rxlib  88b5264ab22dec7287d234726ecec26e94200985
13:12:59 [INFO] [github] fetching https://github.com/RockyLOMO/rxlib @ 88b5264ab22dec7287d234726ecec26e94200985
13:12:59 [INFO] [1] start  https://github.com/RockyLOMO/rxlib  ee74de713ebf246e533ff81321686a1708240643
13:12:59 [INFO] [github] fetching https://github.com/RockyLOMO/rxlib @ ee74de713ebf246e533ff81321686a1708240643
13:12:59 [INFO] [2] start  https://github.com/RockyLOMO/rxlib  5d9bc8d39c7cf18f4ffd3d31182c646ffeeba557
13:12:59 [INFO] [github] fetching https://github.com/RockyLOMO/rxlib @ 5d9bc8d39c7cf18f4ffd3d31182c646ffeeba557
13:12:59 [INFO] [3] start  https://github.com/cgeo/cgeo  509883af5b6db985cac340a5d9539ca6a54173e3
13:12:59 [INFO] [github] fetching https://github.com/cgeo/cgeo @ 509883af5b6db985cac340a5d9539ca6a54173e3
13:12:59 [INFO] [4] start  https://github.com/cgeo/cgeo  8f983a7b79e613d5e8f9be4f4d0402feecead6f8
13:12:59 [INFO] [github] fetching https://github.c

150 downloads queued (50 repos × 3 snapshots)


13:13:00 [INFO] [github] commit info:
hash:    88b5264ab22dec7287d234726ecec26e94200985
  date:    2021-12-27 15:31:59 +0800
  author:  Rocky.℞ <1748083+RockyLOMO@users.noreply.github.com>
  subject: up
13:13:00 [INFO] [github] checking out FETCH_HEAD …
13:13:00 [INFO] [github] commit info:
hash:    5d9bc8d39c7cf18f4ffd3d31182c646ffeeba557
  date:    2026-04-03 03:18:39 +0000
  author:  github-actions[bot] <github-actions[bot]@users.noreply.github.com>
  subject: chore: release v2.21.11 [skip ci]
13:13:00 [INFO] [github] checking out FETCH_HEAD …
13:13:00 [INFO] [github] commit info:
hash:    ee74de713ebf246e533ff81321686a1708240643
  date:    2024-12-09 18:40:11 +0800
  author:  ℞ <ilovehaley.kid@gmail.com>
  subject: Update Socks5CommandRequestHandler.java
13:13:00 [INFO] [github] checking out FETCH_HEAD …
13:13:00 [INFO] [github] done in 1.3s → ../output-final/sources/2021_RockyLOMO_rxlib
13:13:00 [INFO] [0] done in 1.3s  [github]  https://github.com/RockyLOMO/rxlib  88b5264ab22dec7

CancelledError: 

In [ ]:
for task in results:
    source = task.source.value if task.source else "—"
    if task.status == TaskStatus.DONE:
        print(f"  OK   [{source:<9}] {task.url}")
        print(f"         → {task.output_dir}")
    else:
        print(f"  FAIL [{source:<9}] {task.url}")
        print(f"         ✗ {task.error}")

done   = sum(1 for t in results if t.status == TaskStatus.DONE)
failed = len(results) - done
print(f"\n{done}/{len(results)} succeeded  —  {failed} failed")